# Semantic Matching with AutoGluon

Semantic matching answers the question: *"How similar are these two things?"* Unlike classification, there is often no pre-defined set of categories — instead you work with **embedding vectors** and similarity scores.

Use cases:
- Semantic search (find documents similar to a query)
- Duplicate detection
- Image-text retrieval (CLIP-style)
- Product matching across catalogues

This notebook covers:
1. Text-to-text semantic similarity (zero-shot)
2. Extracting embeddings and computing cosine similarity
3. Building a nearest-neighbour search index
4. Image-text matching with CLIP
5. Fine-tuning on labelled pairs
6. Practical gotchas

In [ ]:
import autogluon
print('AutoGluon version:', autogluon.__version__)

## 1. Text Semantic Similarity — Zero-Shot

AutoGluon can extract dense embeddings from text using pre-trained models **without any training data**. The embeddings encode semantic meaning — similar sentences have similar vectors.

In [ ]:
import pandas as pd
import numpy as np
from autogluon.multimodal import MultiModalPredictor

# Create a corpus of sentences to search over
corpus = pd.DataFrame({'text': [
    'The dog ran across the park.',
    'A puppy was playing in the garden.',
    'The cat sat on the windowsill.',
    'She adopted a kitten last week.',
    'Machine learning models learn patterns from data.',
    'Neural networks are inspired by the human brain.',
    'The stock market fell sharply today.',
    'Investors are worried about inflation.',
    'The chef prepared a delicious pasta dish.',
    'Italian cuisine is famous around the world.',
]})

# Use feature_extraction problem type for embedding-based tasks
embedder = MultiModalPredictor(
    problem_type='feature_extraction',
    hyperparameters={
        'model.hf_text.checkpoint_name': 'sentence-transformers/all-MiniLM-L6-v2',
    },
)

# Extract embeddings — returns a numpy array of shape (n_samples, embedding_dim)
corpus_embeddings = embedder.extract_embedding(corpus)
print('Embedding shape:', corpus_embeddings.shape)

## 2. Cosine Similarity

> **Important:** Embeddings are NOT unit-normalized by default. Normalize to unit length before computing cosine similarity, or use `sklearn`'s `cosine_similarity` which handles this.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute pairwise similarity matrix
sim_matrix = cosine_similarity(corpus_embeddings)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(sim_matrix, cmap='RdYlGn', vmin=0, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(corpus)))
ax.set_yticks(range(len(corpus)))
ax.set_xticklabels([t[:25] for t in corpus['text']], rotation=45, ha='right', fontsize=8)
ax.set_yticklabels([t[:25] for t in corpus['text']], fontsize=8)
ax.set_title('Pairwise Cosine Similarity')
plt.tight_layout()
plt.show()

## 3. Nearest-Neighbour Semantic Search

In [ ]:
def semantic_search(query: str, corpus_df: pd.DataFrame, embeddings: np.ndarray, top_k: int = 3):
    query_embedding = embedder.extract_embedding(pd.DataFrame({'text': [query]}))
    scores = cosine_similarity(query_embedding, embeddings)[0]
    top_indices = np.argsort(scores)[::-1][:top_k]
    print(f'Query: "{query}"\n')
    for rank, idx in enumerate(top_indices, 1):
        print(f'  #{rank} (score={scores[idx]:.3f}): {corpus_df.iloc[idx]["text"]}')

semantic_search('I got a new pet', corpus)
print()
semantic_search('deep learning and AI', corpus)

## 4. Image-Text Matching (CLIP)

CLIP-based models embed both images and text into the **same** vector space, enabling cross-modal retrieval: given a text query, find the best matching images (or vice versa).

This works **zero-shot** — no training needed.

In [ ]:
import os, zipfile, urllib.request

# Reuse images from the shopee dataset (if available) or download a small sample
IMG_DIR = './shopee_data/data/test'
if not os.path.exists(IMG_DIR):
    DATA_URL = 'https://autogluon.s3.amazonaws.com/datasets/shopee-iet.zip'
    urllib.request.urlretrieve(DATA_URL, './shopee-iet.zip')
    with zipfile.ZipFile('./shopee-iet.zip', 'r') as z:
        z.extractall('.')
    os.rename('shopee-iet', 'shopee_data')
    os.remove('./shopee-iet.zip')

# Collect a handful of images
image_paths = []
for cls_dir in sorted(os.listdir(IMG_DIR)):
    cls_path = os.path.join(IMG_DIR, cls_dir)
    if os.path.isdir(cls_path):
        for f in sorted(os.listdir(cls_path))[:2]:  # 2 images per class
            if f.endswith(('.jpg', '.png')):
                image_paths.append(os.path.abspath(os.path.join(cls_path, f)))

print(f'Collected {len(image_paths)} images')

In [ ]:
# Build a CLIP-based predictor for image-text similarity
clip_predictor = MultiModalPredictor(
    problem_type='image_text_similarity',
)

# Extract image embeddings from the corpus
image_corpus_df = pd.DataFrame({'image': image_paths})
image_embeddings = clip_predictor.extract_embedding(image_corpus_df)
print('Image embedding shape:', image_embeddings.shape)

In [ ]:
from PIL import Image as PILImage

def image_text_search(text_query: str, image_paths: list, image_embeddings: np.ndarray, top_k: int = 3):
    """Find images that best match a text description."""
    text_embedding = clip_predictor.extract_embedding(pd.DataFrame({'text': [text_query]}))
    scores = cosine_similarity(text_embedding, image_embeddings)[0]
    top_indices = np.argsort(scores)[::-1][:top_k]

    fig, axes = plt.subplots(1, top_k, figsize=(12, 4))
    fig.suptitle(f'Query: "{text_query}"', fontsize=12)
    for ax, idx in zip(axes, top_indices):
        ax.imshow(PILImage.open(image_paths[idx]))
        ax.set_title(f'Score: {scores[idx]:.3f}', fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

image_text_search('a piece of clothing', image_paths, image_embeddings)
image_text_search('electronic device or gadget', image_paths, image_embeddings)

## 5. Fine-Tuning on Labelled Pairs

If you have labelled pairs (e.g., `(sentence_A, sentence_B, is_similar)`), you can fine-tune the model to improve domain-specific similarity.

In [ ]:
from autogluon.tabular import TabularDataset

# SNLI (Stanford Natural Language Inference) subset — sentence pairs with similarity labels
train_pairs = TabularDataset(
    'https://autogluon.s3.amazonaws.com/datasets/text_semantic_search/train.csv'
)
val_pairs = TabularDataset(
    'https://autogluon.s3.amazonaws.com/datasets/text_semantic_search/dev.csv'
)

print('Train pairs shape:', train_pairs.shape)
print('Columns:', train_pairs.columns.tolist())
train_pairs.head()

In [ ]:
# Fine-tune a text similarity model
ft_predictor = MultiModalPredictor(
    problem_type='text_similarity',
    label='label',
    query='sentence1',    # column name for the first item in each pair
    response='sentence2', # column name for the second item in each pair
    path='./ag_semantic_matching_model',
)
ft_predictor.fit(
    train_data=train_pairs,
    tuning_data=val_pairs,
    time_limit=120,
)
print('Fine-tuning complete.')

In [ ]:
# Predict similarity scores on new pairs
new_pairs = pd.DataFrame({
    'sentence1': ['I love playing football.', 'The cat is sleeping.'],
    'sentence2': ['Football is my favourite sport.', 'A dog is running outside.'],
})

similarity_scores = ft_predictor.predict(new_pairs)
for i, (_, row) in enumerate(new_pairs.iterrows()):
    print(f'  A: "{row["sentence1"]}"')
    print(f'  B: "{row["sentence2"]}"')
    print(f'  Similarity: {similarity_scores[i]:.3f}\n')

## ⚠️ Practical Gotchas

| Gotcha | What Happens | Fix |
|--------|-------------|-----|
| **Embeddings are not unit-normalized** | Dot product ≠ cosine similarity | Normalize first: `emb / np.linalg.norm(emb, axis=1, keepdims=True)`, or use `cosine_similarity()` |
| **`feature_extraction` vs `text_similarity`** | Different problem types for different tasks | Use `feature_extraction` for embeddings + manual similarity; use `text_similarity` for supervised pair matching |
| **CLIP is English-centric** | Poor results on non-English text | Use multilingual CLIP variants like `laion/CLIP-ViT-B-32-xlm-roberta-large` |
| **Scaling to large corpora** | Computing pairwise cosine similarity for 100k items is O(n²) | Use an approximate nearest-neighbour library like `faiss` or `hnswlib` for large corpora |
| **query/response column names required for pair tasks** | `KeyError` at fit time | Pass `query='col_a'` and `response='col_b'` when initialising the predictor |
| **Similarity score range varies by model** | Some models output `[0, 1]`, others `[-1, 1]` | Check by computing similarity between identical sentences (should be ~1.0) |
| **Zero-shot CLIP similarity is not calibrated** | Score of 0.3 may mean "very similar" depending on the model | Always interpret scores relatively (rank-order), not absolutely |